# Warm-up: entity linking on one mention per type

A smoke test before Stage 1 of `entity_linking_evaluation_spec.md`, **not** part of the
evaluation itself. It takes one mention per entity type out of `testset_flat_nt.csv` and
runs the current linker on it, printing what was retrieved, what the SPARQL `ASK` check
kept, and what the LLM finally chose.

The point is to eyeball the pipeline end to end and sanity-check the ECKs against real
collection strings before committing to frozen pools and annotation.

**System under test:** `TypedRAGEntityLinker` — resolves its own ECK from `entity_type`,
validation on by default. This is `S3` (neuro-symbolic) in the spec's system table.

Two defects the spec flags as "fix before running" are already fixed in the linker:

- §5 — a SPARQL failure no longer counts as a rejection. `run_sparql_query` retries with
  backoff, and a candidate whose check could not be evaluated is **kept** and flagged,
  not silently discarded.
- §7.3 — `RAGEntityLinker.__init__` now passes the *resolved* key to `genai.Client`, so the
  `GEMINI_API_KEY` fallback fires.

Still outstanding for Stage 5: `link()` collapses several situations into `None`. This
notebook reconstructs the `outcome` taxonomy from the stage counts as a preview of the
instrumentation that `run_systems.py` will need.

## 1. Setup

`bmeapi/src` goes on the path directly rather than importing `bmeapi.src.linker`, because
`bmeapi/__init__.py` pulls in FastAPI, which the `lechoes` env does not have. `linker.py`
supports both import styles for exactly this reason.

In [1]:
import csv
import os
import sys
import time
from collections import OrderedDict
from pathlib import Path

import pandas as pd

# Locate bmeapi/src whether the kernel starts in this folder or at the repo root.
HERE = Path.cwd()
SRC = next(
    (p / "bmeapi" / "src" for p in [HERE, *HERE.parents] if (p / "bmeapi" / "src").is_dir()),
    None,
)
assert SRC is not None, f"could not find bmeapi/src from {HERE}"

EVAL_DIR = SRC.parent / "eval" / "entity-linking"
TESTSET = EVAL_DIR / "testset_flat_nt.csv"
for path in (str(SRC), str(EVAL_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

from entichecks import get_eck, normalise_type          # noqa: E402
from linker import TypedRAGEntityLinker, validate_entities_batch  # noqa: E402

# Reuse Stage 1's extraction rules rather than restating them (spec §3.1).
from build_annotation_sheet import COLUMN_TO_TYPE, MULTIVALUE_SEP, norm  # noqa: E402

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

print(f"src     : {SRC}")
print(f"testset : {TESTSET}  (exists={TESTSET.exists()})")
print(f"types   : {len(COLUMN_TO_TYPE)} -> {', '.join(COLUMN_TO_TYPE.values())}")

src     : /Users/jacodb/Documents/projects/linked-echoes/bmeapi/src
testset : /Users/jacodb/Documents/projects/linked-echoes/bmeapi/eval/entity-linking/testset_flat_nt.csv  (exists=True)
types   : 10 -> music artist, music work, music album, event, venue, place, country, music genre, material, colour


## 2. Extract mentions

Same rules as Stage 1: split on `|`, NFKC + quote normalisation + whitespace collapse,
**no** spelling or case correction — surface errors like *Freddy Mercury* and *Edge Road*
are part of the linking task.

Read with `csv.DictReader` rather than `pd.read_csv` deliberately: `build_annotation_sheet.py`
parses it that way, and identical parsing means the mentions here are the same set Stage 1
will produce. pandas is used for display only.

The row's `description` is carried along as the disambiguation context, which is what
`populate.py` passes as `context_query` in production.

In [2]:
def extract_mentions(csv_path):
    """[(entity_type, label, context, row_no)] in document order, one per occurrence."""
    out = []
    with open(csv_path, newline="", encoding="utf-8") as fh:
        for row_no, row in enumerate(csv.DictReader(fh), start=1):
            context = norm(row.get("description") or "")
            for column, entity_type in COLUMN_TO_TYPE.items():
                for part in (row.get(column) or "").split(MULTIVALUE_SEP):
                    label = norm(part)
                    if label:
                        out.append((entity_type, label, context, row_no))
    return out


mentions = extract_mentions(TESTSET)

counts = pd.DataFrame(
    [
        {
            "entity_type": t,
            "distinct": len({lab.casefold() for et, lab, _, _ in mentions if et == t}),
            "instances": sum(1 for et, _, _, _ in mentions if et == t),
        }
        for t in COLUMN_TO_TYPE.values()
    ]
)
print(f"{len(mentions)} mention instances in {TESTSET.name}")
counts

194 mention instances in testset_flat_nt.csv


,entity_type,distinct,instances
0,music artist,44,78
1,music work,23,29
2,music album,10,14
3,event,2,4
4,venue,8,9
5,place,4,6
6,country,5,10
7,music genre,10,28
8,material,3,9
9,colour,6,7


## 3. Pick one mention per type

First occurrence in document order — deterministic, and no cherry-picking of easy cases.
It happens to surface several of the hard cases the spec calls out in §6 and §9, including
ones that should come back **NIL**: those are the interesting ones, because a linker that
never abstains scores badly on `SL` (spurious link).

Set `PICK_OVERRIDES[<type>] = "<label>"` to probe a different mention.

In [3]:
PICK_OVERRIDES = {
    # "venue": "UFO Club",
    # "music artist": "Cream",
}

# Cases the spec singles out, so the output can be read with the right expectation.
SPEC_NOTES = {
    "new orleans joys": "§6: expect NIL / no_entity — no Wikidata item for this concert",
    "metropolitan club, edge road": "§6/§9: 'Edge Road' is found noise for Edgware Road; bare 'Metropolitan Club' is too_vague",
    "cream": "§9: cross-domain homonym",
    "suede": "§9: cross-domain homonym",
    "queen": "§9: cross-domain homonym",
    "band aid": "§9: supergroup vs recording project",
    "freight train": "§9: title that reads as ordinary language",
    "blue monday": "§9: work/release homonym",
    "white room": "§9: title that reads as ordinary language",
    "apache": "§9: title that reads as ordinary language",
    "foundations": "§9: title that reads as ordinary language",
    "ken colyer": "§9: person vs eponymous ensemble (Ken Colyer's Jazzmen)",
    "the detours": "§9: alias drift -> The High Numbers -> The Who",
    "railway hotel": "§6: expect NIL / out_of_scope — generic venue name",
    "white hart hotel": "§6: expect NIL / no_entity",
    "torquay town hall": "§9: generic venue name; VENUE_ECK may not reach town halls",
    "freddy mercury": "§9: found misspelling of Freddie Mercury",
    "middle earth underground club": "§9: fictional/cultural collision",
    "ufo club": "§9: collides with the band UFO",
    "colour by numbers": "§6: work/release homonym, alternates expected",
}

picks = OrderedDict()
for etype in COLUMN_TO_TYPE.values():
    want = PICK_OVERRIDES.get(etype)
    match = next(
        (m for m in mentions
         if m[0] == etype and (want is None or m[1].casefold() == want.casefold())),
        None,
    )
    if match is None:
        print(f"!! no mention for {etype!r}"
              + (f" matching {want!r}" if want else ""))
        continue
    picks[etype] = {"label": match[1], "context": match[2], "row": match[3]}

pd.DataFrame([
    {
        "entity_type": t,
        "label": p["label"],
        "row": p["row"],
        "eck": normalise_type(t) if get_eck(t) else "** MISSING **",
        "spec_note": SPEC_NOTES.get(p["label"].casefold(), ""),
    }
    for t, p in picks.items()
])

,entity_type,label,row,eck,spec_note
0,music artist,Ken Colyer,1,music_artist,§9: person vs eponymous ensemble (Ken Colyer's Jazzmen)
1,music work,Freight Train,5,music_work,§9: title that reads as ordinary language
2,music album,Disraeli Gears,21,music_album,
3,event,New Orleans Joys,1,event,§6: expect NIL / no_entity — no Wikidata item for this c...
4,venue,"Metropolitan Club, Edge Road",4,venue,§6/§9: 'Edge Road' is found noise for Edgware Road; bare...
5,place,London,4,place,
6,country,Germany,2,country,
7,music genre,Jazz,1,genre,
8,material,Paper,28,material,
9,colour,Turquoise,10,colour,


## 4. Build the linker

Paste the key in the cell below, or leave it blank to fall back to a `GEMINI_API_KEY`
environment variable. Worth clearing it again before committing, since notebook outputs
and source both get saved.

`strict_types=True` makes an unmapped entity type raise instead of quietly skipping
validation, which is what an evaluation harness wants: a silent fallback to unvalidated
behaviour would turn `S3` into `S2` and corrupt the comparison.

In [ ]:
GEMINI_API_KEY = ""  # paste your key here (or leave blank to use the env var)

API_KEY = GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY", "")
assert API_KEY, "set GEMINI_API_KEY above, or export it and restart the kernel"

TOP_K = 10  # spec §7.2 headline k

linker = TypedRAGEntityLinker(
    generator_model="gemini-2.5-flash",
    generator_api_key=API_KEY,
    strict_types=True,
    cache_path=None,  # no caching: we want to watch every call
)

In [6]:


API_KEY = GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY", "")
assert API_KEY, "set GEMINI_API_KEY above, or export it and restart the kernel"

TOP_K = 10  # spec §7.2 headline k

linker = TypedRAGEntityLinker(
    generator_model="gemini-2.5-flash",
    generator_api_key=API_KEY,
    strict_types=True,
    cache_path=None,  # no caching: we want to watch every call
)

LLM Client initialised with model 'gemini-2.5-flash'.


## 5. Probe one mention

`probe` shows the three stages separately so the pruning is visible:

1. **retrieve** — `wikidata_entity_lookup` top-`k`
2. **validate** — the type's `ASK` check over those candidates, batched into one request
3. **select** — `link()`, the real public entry point, for the authoritative answer

Stage 2 is computed here purely so the counts and per-candidate verdicts can be displayed;
`link()` repeats it internally. That duplication is fine for ten mentions but is exactly
what Stage 5 must avoid — `run_systems.py` should emit `n_retrieved` / `n_validated`
from inside a single pass.

A candidate is dropped only on an explicit `False`. A missing or `None` verdict means the
check could not be evaluated, and is shown as `ERR` and **kept** (spec §5).

In [7]:
WD = "https://www.wikidata.org/wiki/"
MARK = {True: "keep", False: "  . ", None: "ERR "}


def probe(entity_type, label, context, k=TOP_K, show_candidates=True):
    eck = linker.resolve_eck(entity_type)
    t0 = time.time()

    retrieved = linker._search_candidates(label, top_k=k) or []
    ids = [c["wikidata_id"] for c in retrieved]
    # None => whole batch failed; treat every verdict as unevaluable, not as False.
    verdicts = validate_entities_batch(ids, eck) or {}
    kept = [c for c in retrieved if verdicts.get(c["wikidata_id"]) is not False]
    n_error = sum(1 for i in ids if verdicts.get(i) is None)

    chosen = linker.link(
        search_term=label,
        context_query=context,
        entity_type=entity_type,
        top_k=k,
        validate=True,
        verbose=False,
    )
    elapsed = time.time() - t0

    if not retrieved:
        outcome = "abstain_no_candidates"
    elif not kept:
        outcome = "abstain_all_filtered"
    elif chosen is None:
        outcome = "abstain_llm"
    else:
        outcome = "linked"

    print(f"\n{'=' * 78}\n{entity_type.upper()}  |  {label!r}")
    note = SPEC_NOTES.get(label.casefold())
    if note:
        print(f"  note   : {note}")
    print(f"  context: {context[:100]}{'...' if len(context) > 100 else ''}")
    print(f"{'-' * 78}")
    print(f"  retrieved {len(retrieved)} -> validated {len(kept)}"
          f"{f' ({n_error} unevaluable)' if n_error else ''}   [{elapsed:.1f}s]")

    if show_candidates:
        for i, c in enumerate(retrieved, start=1):
            star = " <-- picked" if chosen and c["wikidata_id"] == chosen["wikidata_id"] else ""
            desc = (c.get("description") or "")[:50]
            print(f"    {MARK[verdicts.get(c['wikidata_id'])]} {i:>2}. "
                  f"{c['wikidata_id']:<11} {(c['label'] or '')[:26]:<27} {desc}{star}")

    print(f"{'-' * 78}")
    if chosen:
        print(f"  LINKED -> {chosen['label']} ({chosen['wikidata_id']})")
        print(f"            {WD}{chosen['wikidata_id']}")
    else:
        print(f"  NO LINK  ({outcome})")

    return {
        "entity_type": entity_type, "label": label,
        "n_retrieved": len(retrieved), "n_validated": len(kept), "n_error": n_error,
        "predicted_qid": chosen["wikidata_id"] if chosen else None,
        "predicted_label": chosen["label"] if chosen else None,
        "outcome": outcome, "latency_s": round(elapsed, 1),
    }

## 6. Run all ten

Roughly 2 Wikidata API calls, 2–4 SPARQL requests and 1 Gemini call per mention. WDQS is
throttled to ~1 req/s inside `run_sparql_query`, so expect a minute or two.

In [8]:
results = []
for etype, p in picks.items():
    try:
        results.append(probe(etype, p["label"], p["context"]))
    except Exception as exc:  # keep going so one failure does not lose the whole run
        print(f"\n!! {etype} / {p['label']!r} raised {type(exc).__name__}: {exc}")
        results.append({
            "entity_type": etype, "label": p["label"], "n_retrieved": None,
            "n_validated": None, "n_error": None, "predicted_qid": None,
            "predicted_label": None, "outcome": f"exception:{type(exc).__name__}",
            "latency_s": None,
        })

print(f"\n\ndone: {len(results)} mentions")


MUSIC ARTIST  |  'Ken Colyer'
  note   : §9: person vs eponymous ensemble (Ken Colyer's Jazzmen)
  context: Souvenir programme for the 'New Orleans Joys' concert featuring Ken Colyer's Jazzmen and Acker Bilk'...
------------------------------------------------------------------------------
  retrieved 3 -> validated 3   [5.6s]
    keep  1. Q1738588    Ken Colyer                  British musician (1928–1988) <-- picked
    keep  2. Q121602937  Ken Colyer's Skiffle Group  British skiffle group
    keep  3. Q121557449  Ken Colyer's Jazzmen        
------------------------------------------------------------------------------
  LINKED -> Ken Colyer (Q1738588)
            https://www.wikidata.org/wiki/Q1738588

MUSIC WORK  |  'Freight Train'
  note   : §9: title that reads as ordinary language
  context: Chas McDevitt - Freight Train 10" LP
------------------------------------------------------------------------------
  retrieved 10 -> validated 2   [17.0s]
      .   1. Q954927     freight

## 7. Summary

`n_retrieved -> n_validated` is the mean pruning of spec §8.2 metric 6. Judge each row by
hand against Wikidata — there is no gold standard yet, that is Stage 4.

In [9]:
summary = pd.DataFrame(results)[[
    "entity_type", "label", "n_retrieved", "n_validated", "n_error",
    "outcome", "predicted_label", "predicted_qid", "latency_s",
]]

scored = summary.dropna(subset=["n_retrieved"])
if not scored.empty:
    print(f"mean pruning : {scored['n_retrieved'].mean():.1f} -> {scored['n_validated'].mean():.1f}")
    print(f"linked       : {(summary['outcome'] == 'linked').sum()}/{len(summary)}")
    print(f"unevaluable  : {int(scored['n_error'].sum())} candidate checks")
    print("outcomes     : "
          + ", ".join(f"{k}={v}" for k, v in summary["outcome"].value_counts().items()))

summary

mean pruning : 6.4 -> 2.2
linked       : 8/10
unevaluable  : 0 candidate checks
outcomes     : linked=8, abstain_no_candidates=2


,entity_type,label,n_retrieved,n_validated,n_error,outcome,predicted_label,predicted_qid,latency_s
0,music artist,Ken Colyer,3,3,0,linked,Ken Colyer,Q1738588,5.6
1,music work,Freight Train,10,2,0,linked,Freight Train,Q106634082,17.0
2,music album,Disraeli Gears,1,1,0,linked,Disraeli Gears,Q269211,3.0
3,event,New Orleans Joys,0,0,0,abstain_no_candidates,NaN,NaN,0.4
4,venue,"Metropolitan Club, Edge Road",0,0,0,abstain_no_candidates,NaN,NaN,0.4
5,place,London,10,5,0,linked,London,Q84,42.7
6,country,Germany,10,3,0,linked,Germany,Q183,11.4
7,music genre,Jazz,10,2,0,linked,jazz,Q8341,5.6
8,material,Paper,10,3,0,linked,paper,Q11472,5.9
9,colour,Turquoise,10,3,0,linked,turquoise,Q5960345,8.6


In [10]:
out_path = EVAL_DIR / "warmup_results.csv"
summary.to_csv(out_path, index=False)
print(f"wrote {out_path}")

wrote /Users/jacodb/Documents/projects/linked-echoes/bmeapi/eval/entity-linking/warmup_results.csv


## 8. What to look for

Before moving to Stage 1, read the output above for:

- **Over-pruning.** `n_validated == 0` on a mention whose gold entity was clearly in the
  retrieved list is a validator false rejection — spec §8.2 metric 5, the number that must
  be reported honestly. Venues are the predicted sore point: `VENUE_ECK` reaches venues and
  recording studios, and *Torquay Town Hall* / *White Hart Hotel* / *Railway Hotel* may well
  not be typed as either in Wikidata.
- **Under-pruning.** `n_validated == n_retrieved` means the check did no work; verify that
  is right for the type rather than a sign the ECK is too loose.
- **Correct abstention.** *New Orleans Joys* should come back with no link. If it links to
  something plausible-looking, that is a spurious link (`SL`) and exactly the failure mode
  validation is meant to prevent.
- **Work/album routing.** Spec §10 notes these are not disjoint: a single that is also a
  song can validate under both `MUSIC_WORK_ECK` and `MUSIC_ALBUM_ECK`. *Blue Monday* and
  *Colour By Numbers* are the mentions to test that on.
- **`ERR` rows.** A candidate whose check could not be evaluated is kept, not dropped.
  Non-zero counts mean WDQS was flaky, so re-run before drawing conclusions.

Anything that looks like an ECK defect goes through the §5 revision protocol — record it,
fix it in `entichecks.py`, re-run `test_linker.py`, and bump `eck_version`. Do **not** tune
quietly against these examples once the pools are frozen.

### Next

Stage 1 expects `gold_objects.csv`; `testset_flat_nt.csv` is the flat export used here. The
spec's §13 checklist starts with regenerating `annotation_sheet.csv` from the real object
CSV, which should give 115 distinct mentions over 194 instances — compare against the
counts in section 2 to confirm this file is the same underlying set.